<a href="https://colab.research.google.com/github/estevaosmith/Portifolio/blob/main/PROJETO_Churn_Prediction_com_RandomForestClassificator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Churn Prediction

*A taxa de abandono, conhecida como churn rate, reflete a porcentagem de clientes que deixam sua base de usu?rios. Em plataformas como Spotify ou Netflix, ela indica quantas assinaturas foram canceladas. Sua an?lise ? crucial para a gest?o, revelando poss?veis problemas a serem resolvidos ao longo do tempo.*

<p align=center>
<img src="https://img.freepik.com/fotos-gratis/jovem-adulto-pressionando-o-botao-da-campainha_23-2149586572.jpg?t=st=1710190440~exp=1710194040~hmac=5fb8d688384c87a5af0c9814e23ae1710f3b5775b52022abd1e9d4f1ff6a2323&w=996" width="60%"></p>

Al?m disso, o churn pode antecipar cancelamentos e orientar a??es espec?ficas para reter clientes. ? vital destacar que o Custo de Aquisi??o de Cliente (CAC) geralmente supera o custo de manuten??o, tornando essencial controlar o churn rate, pois altas taxas s?o indesej?veis.







## Aquisi??o dos Dados

Os dados utilizados neste projeto foram inicialmente disponibilizados na [plataforma de ensino da IBM Developer](https://developer.ibm.com/technologies/data-science/patterns/predict-customer-churn-using-watson-studio-and-jupyter-notebooks/#) e tratam de um problema comum enfrentado por empresas de telecomunica??es. O conjunto de dados completo est? acess?vel atrav?s [deste link](https://raw.githubusercontent.com/carlosfab/dsnp2/master/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv).

Embora n?o haja informa??es explicitamente fornecidas, os nomes das colunas permitem uma compreens?o do problema em quest?o.

In [ ]:
# instalar Pandas Profiling
!pip install scikit-plot -q
!pip install imbalanced-learn -q

In [ ]:
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)

# importar os pacotes necess?rios
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, classification_report
from scikitplot.metrics import plot_confusion_matrix, plot_roc
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# random seed
np.random.seed(42)

# importar os dados
DATA_PATH = "https://raw.githubusercontent.com/carlosfab/dsnp2/master/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

# ver as 5 primeiras entradas
df.head()

In [ ]:
print("Linhas:\t\t{}".format(df.shape[0]))
print("Colunas:\t{}".format(df.shape[1]))

Como podemos ver, o dataset possui 7043 e 21 colunas.



## An?lise explorat?ria

Agora, preciso realizar uma an?lise explorat?ria para garantir que os dados estejam devidamente tratados antes de implementar o modelo de Machine Learning.

In [ ]:
print("Dados ausentes (total):\t{}\n".format(df.isnull().sum().values.sum()))
print(df.isnull().sum() / df.shape[0])
print("\nNomes das features:\n{}\n".format(df.columns.values))
print("Valores ?nicos (por coluna):\n{}\n".format(df.nunique()))
print("Tipos de Vari?veis (por coluna)\n{}".format(df.dtypes))

Algumas observa??es importantes foram feitas durante esta fase inicial de conscientiza??o:
1. A vari?vel alvo est? na coluna Churn.
2. A coluna TotalCharges deveria ser do tipo float, mas atualmente ? uma string.
3. Muitas caracter?sticas precisam ser convertidas em valores num?ricos.


## Tratar os dados
Ap?s esta breve an?lise, ? necess?rio preparar os dados. Vou utilizar o *LabelEncoder* para transformar todas as categorias em 0 ou 1, tornando assim o modelo de Machine Learning vi?vel.

In [ ]:
def converter_str_float(entrada):

    try:
        return float(entrada)
    except ValueError:
        return np.nan


df_clean = df.copy()

# remover coluna costumerID
df_clean.drop("customerID", axis=1, inplace=True)

# converter as strings de TotalCharges para float
df_clean["TotalCharges"] = df_clean["TotalCharges"].apply(converter_str_float)

# verificar entradas ausentes
print("Dados ausentes em 'TotalCharges' (antes):\t{}\n".format(df_clean["TotalCharges"].isnull().sum()))

# substituir os dados ausentes pelo valor da mediana
TotalChargesMedian = df_clean.TotalCharges.median()
df_clean["TotalCharges"].fillna(TotalChargesMedian, inplace=True)

# pr? processar as vari?veis
binary_var = df_clean.nunique()[df_clean.nunique() == 2].keys().tolist()
num_var = [col for col in df_clean.select_dtypes(['int', 'float']).columns.tolist() if col not in binary_var]
cat_var = [col for col in df_clean.columns.tolist() if col not in binary_var + num_var]

# criar uma c?pia do dataframe
df_proc = df_clean.copy()

# Label Encoding para as vari?veis Bin?rias
le = LabelEncoder()
for i in binary_var:
    df_proc[i] = le.fit_transform(df_proc[i])

# Encoding para colunas categ?ricas com m?ltiplas classes
df_proc = pd.get_dummies(df_proc, columns=cat_var)

# ver as 5 primeiras entradas do novo dataframe
df_proc.head()


Podemos observar que a classe "No" ocorre com uma frequ?ncia significativamente maior do que a classe "Yes". Esse fato ser? levado em considera??o durante a constru??o do modelo de machine learning.

In [ ]:
sns.countplot(data=df_clean, x='Churn');

## Modelo de Machine Learning

? fundamental, desde o in?cio, dividir os dados em conjuntos de treinamento e teste. Os dados de teste ser?o reservados exclusivamente para a fase final do projeto, proporcionando uma avalia??o mais precisa e realista.

In [ ]:
X = df_proc.drop('Churn', axis=1)
y = df_proc['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [ ]:
#  padronizar os  dados,
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)

rus = RandomUnderSampler()
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)



Decidi utilizar o **RandomForestClassificator** pois ele possui capacidade de lidar bem com uma ampla variedade de problemas de classifica??o, incluindo conjuntos de dados grandes e complexos, sem a necessidade de ajustes finos extensivos nos hiperpar?metros. Al?m disso, ele ? menos suscet?vel ao overfitting em compara??o com uma ?nica ?rvore de decis?o.Fiz testes com diversos outros modelos; no entanto, o resultado n?o foi muito satisfat?rio.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = RandomForestClassifier(n_estimators=100, random_state=42)  #
model.fit(X_train_resampled, y_train_resampled)

# Fazer previs?es no conjunto de teste
y_pred = model.predict(X_test)

# Avaliar o desempenho do modelo
accuracy = accuracy_score(y_test, y_pred)
print("Acur?cia:", accuracy)

# Imprimir relat?rio de classifica??o
print("\nRelat?rio de Classifica??o:")
print(classification_report(y_test, y_pred))

# imprimir a ?rea sob a curva
print("AUC: {:.4f}\n".format(roc_auc_score(y_test, y_pred)))

# plotar matriz de confus?o
plot_confusion_matrix(y_test, y_pred, normalize=True)
plt.show()



## Conclus?es
Apesar do tratamento de dados ter demandado certo trabalho, o modelo teve um desempenho razoavelmente bom.

Tamb?m ? importante ressaltar que n?o realizei a otimiza??o de hiperpar?metros, pois o modelo que escolhi n?o requer isso explicitamente; no entanto, os resultados poderiam ter sido melhores se a otimiza??o tivesse sido feita, talvez em algum outro modelo.

Em um projeto maior, tamb?m seria adequado aplicar t?cnicas como Feature Selection  e Feature Engineering, afim de melhorar a otimiza??o do modelo.
